# Layers without electrodes

This Notebook demonstrates a simple example how to set up a simulation to simulate semiconductor layers without electrodes (For more information, see [Application Note 6.1 in SIMsalabim Manual](http://simsalabim-online.com/manual/section_layers_without_electrodes.html)).

In [1]:
import os, sys
try:
    import pySIMsalabim as sim
except ImportError: # add parent directory to sys.path if pySIMsalabim is not installed
    sys.path.append('..')
    import pySIMsalabim as sim

from pySIMsalabim.utils.device_parameters import create_contactless_device, process_output_contactless
from pySIMsalabim.utils.general import run_simulation


In [2]:
################# Parameter setup ######################

# Set the path to the simulation setup file
cwd = os.path.abspath('..')
session_path = os.path.join(cwd, 'SIMsalabim','ZimT')
zimt_simulation_setup = os.path.join(session_path, 'simulation_setup.txt')

# Define the output file names
tJFile = 'tJ.dat'
VarFile = 'Var.dat'
tVGFile = 'tVG.txt'
logFile = 'log.txt'

# Create the arguments for the simulation
cmd_pars = [{'par':'dev_par_file','val':zimt_simulation_setup},
                {'par':'tVGFile','val':tVGFile},
                {'par':'tJFile','val':tJFile},
                {'par':'varFile','val':VarFile},
                {'par':'logFile','val':logFile}]

# Specific parameters for simulating semiconductor devices without electrodes.
simulation_type = 'zimt'  # 'simss' or 'zimt'
contactless_device = True # If True, use a contactless device architecture, which adds blocking layers to the simulation setup. 
format_output = True # Whether to format the output files to remove all entries and elements related to the blocking layers and update the layer indices back to their original values

# Set a layer property, whose layer index must match the original device architecture, as it will be reassigned automatically upon addition of the blocking layers.
cmd_pars.append({'par':'l1.N_t_bulk','val':'5E20'}) # TEMPORARY

################# Run the simulation ######################

## Create the contactless device ##
returncode, cmd_pars, block_layer_file = create_contactless_device(session_path, zimt_simulation_setup, cmd_pars = cmd_pars)

## Run a simulation or experiment with the contactless device architecture ##
if returncode == 0:
    returncode, mess = run_simulation(simulation_type, cmd_pars, session_path)
else:
    print(f"Failed to create contactless device architecture. Return code: {returncode}. No simulation was run.")

## Format the output files (optional) ##
# This removes all entries and elements related to the blocking layers and update the layer indices back to their original values
if returncode == 0 and format_output:
    # Note: If the VarFile is specified, formatting it can take a significant amount of time, especially for large simulations, due to the large number of changes needed to be made. If the VarFile is not needed, consider setting it to 'none'
    process_output_contactless(session_path, zimt_simulation_setup, simulation_type, format_output = format_output, 
                                        clean_simulation_setup=False, tJ_file = tJFile, Var_file = VarFile, block_layer_file = block_layer_file)
else:
    print(f"Simulation failed with return code {returncode}. No output files were formatted.")


In [3]:
# Clean up the output files (comment out if you want to keep the output files)
# sim.clean_all_output(session_path)